# Process windows, mask optimization, resist development and maskless interference

Once the aerial image is right, three questions remain: **how robust** is the
print (process window), **what mask** prints the target best (ILT, OPC, SRAFs),
and **how does the resist turn dose into a profile** (bake and development).
The last part covers two lithography techniques without projection optics:
Talbot (displacement) lithography and two-grating EUV interference lithography.

| Topic | Model status |
|---|---|
| Process window / ED analysis | 🔶 constant-threshold resist |
| ILT (adjoint gradient) / OPC | ✅ (ILT proxy gradient 🔶) |
| SRAF insertion | 🔶 rule deck + model print check |
| Level-set development | ✅/🔶 · fast marching ✅ |
| Chemically amplified bake | 🔶 |
| Talbot / DTL / ATL, EUV-IL | 🔶 |

Numbers from ILT, OPC and SRAF depend on the imaging engine settings (kernel
count, defocus model) and on the constant-threshold resist; "prints" means
"crosses that threshold". See [research modules](../../docs/research-modules.md)
and the [capability matrix](../../docs/capability-matrix.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline

## 1. Dose-aware process windows: illumination changes the window

A process window is the set of (focus, dose) settings for which the printed CD
stays within tolerance (here ±10 %). `huv.process_window` builds the
focus-exposure matrix with a constant-threshold resist and analyses it: exposure
latitude (EL), depth of focus (DOF), the largest inscribed rectangle, and the
EL-vs-DOF curve.

Notebook 04 showed that a symmetric dipole keeps a dense grating's contrast
through focus. Here is what that means for the window of 90 nm lines on a 180 nm
pitch (F$_2$, NA 0.75): conventional σ 0.7, annular 0.5-0.8 and a dipole at the
symmetric pole position $\sigma_c = \lambda / (2 p \mathrm{NA})$.

In [ ]:
wl, na, cd, pitch = 157.63, 0.75, 90.0, 180.0
sigma_c = wl / (2 * pitch * na)
illuminations = {
    "conventional σ 0.7": None,
    "annular 0.5-0.8": ("annular", 0.5, 0.8),
    f"dipole σc {sigma_c:.2f}": ("dipole", sigma_c, 0.1, 0.0),
}
doses = list(np.linspace(22.0, 40.0, 19))
focuses = list(np.linspace(-800.0, 800.0, 33))
windows = {}
for name, ill in illuminations.items():
    windows[name] = huv.process_window(cd, pitch, illumination=ill, grid_size=64, pixel_nm=3.0,
                                       doses_mj_cm2=doses, focuses_nm=focuses)
    w = windows[name].dof_at_el(5.0)
    dof = w["dof_nm"] if w else 0.0
    clipped = dof >= focuses[-1] - focuses[0] - 1e-6        # window reaches the scanned range
    print(f"{name:20s} DOF at 5 % EL: {'>=' if clipped else '  '}{dof:5.0f} nm   (status {windows[name].status})")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
viz.plot_ed_window(windows["conventional σ 0.7"], ax=axes[0], title="ED window, conventional σ 0.7")
viz.plot_ed_window(windows[f"dipole σc {sigma_c:.2f}"], ax=axes[1], title=f"ED window, dipole σc {sigma_c:.2f}")
for name, pw in windows.items():
    dof, el = pw.el_vs_dof(n_points=81)
    axes[2].plot(dof, el, label=name)
axes[2].axhline(5.0, ls="--", color="grey", lw=1)
axes[2].set(xlabel="depth of focus (nm)", ylabel="exposure latitude (%)", title="EL vs DOF", ylim=(0, None))
axes[2].legend()
axes[2].grid(alpha=0.3)
fig.tight_layout()
print(f"scanned focus range: {focuses[0]:.0f} .. {focuses[-1]:.0f} nm (a window reaching the edge is clipped)")

The dipole window is several times deeper. The price, not modeled here, is that a
dipole helps one orientation and one pitch: lines in the other direction, or
other pitches, see a much worse source.

## 2. Inverse lithography (ILT)

ILT treats every mask pixel as a free variable and minimizes the difference
between the printed pattern and the target. The cost is a sigmoid "resist" of the
aerial image; its gradient with respect to all mask pixels comes from one
**adjoint** pass through the same SOCS kernels the engine images with (✅,
matches finite differences to about 1e-8). Nonlinear conjugate gradients with an
Armijo line search, total-variation and binarization regularization follow.

Target: a 2 × 2 array of 100 nm contact holes on a 200 nm pitch, F$_2$, NA 0.75,
σ 0.6.

In [ ]:
source = huv.SourceConfig.f2_laser(0.6)
optics = huv.OpticsConfig(numerical_aperture=0.75)
grid = huv.GridConfig(64, 8.0)
target = huv.contact_target(100, 200, 2, grid)
ilt_kw = dict(cost="resist", threshold=0.25, steepness=50, max_iterations=40,
              binarization_weight=0.1, binarization_after=20, snapshot_every=10, max_kernels=8)
ilt = huv.optimize_ilt(source, optics, grid, target, **ilt_kw)
print(f"adjoint: {ilt.termination} after {ilt.iterations} iterations, cost {ilt.cost_history[0]:.4f} -> "
      f"{ilt.cost_history[-1]:.4f}, misprinted pixels (binary mask) {ilt.binary_pattern_error} ({ilt.status})")
fig = viz.plot_ilt_evolution(ilt, target=target, threshold=0.25)

The older **proxy** gradient (🔶) approximated the derivative by a single blur.
It points in a wrong direction often enough that the line search fails early:

In [ ]:
proxy = huv.optimize_ilt(source, optics, grid, target, gradient="proxy", **ilt_kw)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogy(ilt.cost_history, label=f"adjoint ({ilt.termination})")
ax.semilogy(proxy.cost_history, label=f"proxy ({proxy.termination})")
ax.set(xlabel="iteration", ylabel="cost", title="ILT convergence: adjoint vs proxy gradient")
ax.legend()
ax.grid(alpha=0.3, which="both")

## 3. Model-based OPC

Optical proximity correction keeps the mask rectilinear and moves fragment
edges. Each iteration measures the edge-placement error (EPE) of every fragment
on the aerial-image contour and biases the fragment against it. A 90 nm isolated
line at $k_1 \approx 0.43$ pulls back strongly at its ends; OPC pushes the line
ends out until the EPE is below a nanometre.

In [ ]:
grid_opc = huv.GridConfig(64, 12.0)
line = huv.MaskConfig.from_features([(0.0, 0.0, 90.0, 500.0)])      # (x, y, w, h) in nm
thr = huv.dose_to_size_threshold(source, optics, line, grid_opc, 90.0, max_kernels=12)
opc = huv.fragment_opc(source, optics, line, grid_opc, thr, tolerance_nm=0.5,
                       max_epe_tolerance_nm=1.0, max_kernels=12)
print(f"threshold {thr:.3f}; EPE rms per iteration (nm): {np.round(opc.epe_rms_history, 2)}  ({opc.status})")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))
img = np.asarray(opc.final_image)
half = 32 * 12.0
ax0.imshow(img, origin="lower", extent=[-half, half, -half, half], cmap="viridis")
ax0.contour(np.linspace(-half, half, 64), np.linspace(-half, half, 64), img, levels=[thr], colors="w")
for poly in opc.polygons:
    xy = np.array(poly + [poly[0]])
    ax0.plot(xy[:, 0], xy[:, 1], color="orange", lw=1)
ax0.add_patch(plt.Rectangle((-45, -250), 90, 500, fill=False, ls="--", color="red"))
ax0.set(title="corrected mask (orange), print contour (white), target (red)", xlabel="x (nm)", ylabel="y (nm)")
ax1.plot(opc.epe_rms_history, "o-", label="rms")
ax1.plot(opc.epe_max_history, "s-", label="max")
ax1.set(xlabel="iteration", ylabel="|EPE| (nm)", yscale="log", title="OPC convergence")
ax1.legend()
ax1.grid(alpha=0.3, which="both")
fig.tight_layout()

## 4. Sub-resolution assist features (SRAFs)

An isolated line has a poor depth of focus compared with a dense grating.
Scattering bars placed beside it, too narrow to print, make its diffraction
spectrum look more like a dense pattern's. `isolated_line_sraf_study` inserts
bars by a λ/NA-scaled rule deck, checks they do not print at the focus and dose
corners, and compares the DOF (±10 % CD) with and without them.

The benefit depends on the illumination: with annular light the bars extend the
DOF by about 30 %; with a true two-pole dipole the inner bars print, are removed,
and the gain nearly vanishes. SRAF rules are tied to the source.

In [ ]:
studies = {
    "annular 0.5-0.8": huv.isolated_line_sraf_study(80.0),
    "dipole 0.7/0.15": huv.isolated_line_sraf_study(80.0, illumination=("dipole", 0.7, 0.15, 0.0),
                                                    sigma_center=0.7),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (name, st) in zip(axes, studies.items()):
    viz.plot_sraf_dof(st, target_cd_nm=80.0, ax=ax)
    ax.set_title(f"{name}: DOF {st.dof.dof_without_nm:.0f} -> {st.dof.dof_with_nm:.0f} nm (x{st.dof.gain:.2f})")
    print(f"{name:16s} bars kept {len(st.sraf.assists)}, removed {st.sraf.removed}; DOF gain x{st.dof.gain:.2f}")
fig.tight_layout()

## 5. Development: level set vs fast marching, and surface inhibition

Development moves a dissolution front through the exposed resist. Two solvers:

- **fast marching** (✅) solves the eikonal equation for the arrival time of the
  front in a *static* rate field;
- the **level set** (✅/🔶) moves the front step by step and re-evaluates the rate
  *at the front*, which allows rate laws that depend on time or on how much resist
  has dissolved (developer ageing, loading).

On a static problem they must agree. A Gaussian exposed trench, uniform in depth,
developed for 3 s, with and without **surface inhibition** (a top skin that
dissolves 10× slower, decaying over 10 nm). Without inhibition the opening
flares out at the top, where the developer has acted longest; with inhibition
it is narrowest a few nanometres below the surface, the beginning of a T-top.
Both solvers agree to within a grid cell.

In [ ]:
dx, dz = 1.0, 2.0
nx, ny, nz = 128, 2, 50
x = (np.arange(nx) + 0.5) * dx - nx * dx / 2
m = 1 - 0.9 * np.exp(-x ** 2 / (2 * 20.0 ** 2))         # PAC: 1 unexposed, 0.1 at the trench centre
vol = np.ascontiguousarray(np.broadcast_to(m[None, None, :], (nz, ny, nx)))
pac = huv.VolumetricResult.from_array(vol, (-nx * dx / 2, nx * dx / 2), (0.0, ny * dx), (0.0, nz * dz))
resist = huv.ResistConfig()
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (label, ratio, depth) in zip(axes, (("no inhibition", 1.0, 0.0), ("surface inhibition 0.1 / 10 nm", 0.1, 10.0))):
    ls = huv.develop_level_set(pac, resist, 3.0, surface_rate_ratio=ratio, inhibition_depth_nm=depth)
    fmm = huv.develop_fast_marching(pac, resist, dx, dz, surface_rate_ratio=ratio,
                                    inhibition_depth_nm=depth, lateral="periodic")
    viz.plot_development_front({"level set": ls.arrival_times, "fast marching": fmm}, 3.0,
                               ax=ax, title=label)
    ax.set_xlim(-60, 60)
    ax.set_ylim(40, 0)                                   # zoom on the top 40 nm
    print(f"{label:32s} level set: {ls.steps} steps, {ls.reinitializations} reinitializations ({ls.status})")
fig.tight_layout()

### Chemically amplified resist: the quencher sharpens the latent image

In a chemically amplified resist (CAR) the exposure creates acid, and during the
post-exposure bake the acid diffuses and catalytically deprotects the polymer. A
base quencher neutralizes acid where there is little of it, which steepens the
deprotection profile at the cost of dose. `peb_car` (🔶) solves the
reaction-diffusion system (exact local reaction plus lattice-exact diffusion).

In [ ]:
period = 128.0
xs = (np.arange(64) + 0.5) * 2.0
acid0 = 0.6 * 0.5 * (1 + np.tanh(3 * np.cos(2 * np.pi * (xs / period - 0.5))))
latent = huv.VolumetricResult.from_array(np.ascontiguousarray(np.repeat((1 - acid0)[None, None, :], 2, axis=1)),
                                         (0.0, period), (0.0, 4.0), (0.0, 2.0))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, acid0, "k--", label="initial acid")
for q in (0.0, 0.1, 0.2, 0.3):
    car = huv.peb_car(latent, quencher=q)
    ax.plot(xs, 1 - np.asarray(car.protected.values)[0, 0], label=f"deprotected, quencher {q}")
ax.set(xlabel="x (nm)", ylabel="fraction", title="CAR post-exposure bake: quencher sweep")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
print("status", car.status, "-", "; ".join(car.notes))

## 6. Talbot lithography and EUV interference lithography

### The Talbot carpet

A periodic mask lit by a coherent plane wave re-images itself at multiples of the
Talbot length, $z_T = 2 p^2 / \lambda$ in the paraxial limit (the exact
self-imaging distance of the first-order pair is $\lambda / (1 - \sqrt{1 - (\lambda / p)^2})$);
the intensity between the mask and $z_T$ is the "Talbot carpet". Printing at a fixed gap is fragile (the image changes with gap).
**Displacement Talbot lithography** (DTL) averages the exposure over one Talbot
length, giving a gap-independent image; **achromatic Talbot lithography** (ATL)
uses a broadband source beyond the achromatic distance and converges to the same
image. For a 1:1 line grating that image has **half the mask period**: the
averaging removes every interference term except those between the $+m$ and
$-m$ orders. Both are 🔶 (scalar thin-mask orders, normal incidence).

In [ ]:
talbot = huv.simulate_talbot(13.5, 100.0, grating="amplitude", max_order=10, carpet_nz=256,
                             carpet_z_max_nm=2 * 1481.5)
print(f"z_T = {talbot.talbot_length_nm:.1f} nm (2p²/λ = {2 * 100.0**2 / 13.5:.1f} nm), "
      f"exact first-order self-imaging distance {talbot.talbot_length_exact_nm:.1f} nm  ({talbot.status})")
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.4, 1]})
viz.plot_talbot_carpet(talbot, ax=ax0)
xt = np.asarray(talbot.x_nm)
ax1.plot(xt, np.asarray(talbot.coherent_image)[0], label="coherent, at the gap")
ax1.plot(xt, np.asarray(talbot.dtl_image)[0], label="DTL (averaged over z_T)")
atl = huv.simulate_talbot(13.5, 100.0, bandwidth_nm=0.5, exposure="atl")
ax1.plot(np.asarray(atl.x_nm), np.asarray(atl.atl_image)[0], "--", label="ATL, 0.5 nm band")
ax1.set(xlabel="x (nm)", ylabel="intensity", title="Printed image: 100 nm period, 13.5 nm")
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
fig.tight_layout()
print(f"achromatic distance for a 0.5 nm band: {atl.achromatic_distance_nm / 1000:.0f} µm")

### Two-grating EUV interference lithography

Two transmission gratings of period $p$ diffract the ±m orders of a coherent EUV
beam towards each other; where they overlap they form fringes of period
$p / (2m)$, independent of the wavelength. With unequal beam intensities
$I_1 = r I_2$ the fringe visibility drops to $2 \sqrt{r} / (1 + r)$.

In [ ]:
for ratio in (1.0, 0.25):
    il = huv.simulate_euv_il(grating_period_nm=100.0, intensity_ratio=ratio)
    print(f"intensity ratio {ratio:4.2f}: fringe period {il.fringe_period_nm:.1f} nm, visibility {il.visibility:.3f} "
          f"(closed form {2 * np.sqrt(ratio) / (1 + ratio):.3f})  ({il.status})")

Docs: [Research modules](../../docs/research-modules.md) (ILT, OPC, SRAF, multiple
patterning, DSA), [Masks and metrics](../../docs/masks-and-metrics.md) (process
window), [Resist models](../../docs/processes/resist-models.md),
[Talbot](../../docs/processes/talbot.md).